In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# ============================================================
# CELL 0 — Environment Check
# ============================================================
import os, shutil, torch

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

for path in ["/kaggle/working", "/tmp"]:
    total, used, free = shutil.disk_usage(path)
    print(f"{path}  Total:{total/1e9:.1f}G  Used:{used/1e9:.1f}G  Free:{free/1e9:.1f}G")

print(f"\nGPU:  {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

/kaggle/working  Total:21.0G  Used:0.0G  Free:20.9G
/tmp  Total:8656.9G  Used:7346.4G  Free:1310.5G

GPU:  Tesla T4
VRAM: 15.6 GB
CUDA: 12.8


In [3]:
# ============================================================
# CELL 1 — Install Dependencies
# ============================================================
import subprocess
subprocess.run(["pip", "install", "unsloth[colab-new]", "unsloth_zoo", "--quiet"])
subprocess.run(["pip", "install", "--no-deps", "trl>=0.12.0", "peft",
                "accelerate", "bitsandbytes", "--quiet"])
print("✓ Done")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 3.9 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.


✓ Done


In [4]:
# ============================================================
# CELL 2 — Load Gemma 4 E2B Base Model
# ============================================================
from unsloth import FastModel
import torch, os

os.environ["HF_HOME"]            = "/tmp/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf_cache"
os.environ["HF_DATASETS_CACHE"]  = "/tmp/hf_datasets_cache"
os.makedirs("/tmp/hf_cache", exist_ok=True)
os.makedirs("/tmp/hf_datasets_cache", exist_ok=True)

# Load base Gemma 4 E2B — train from scratch with distillation data
# (GGUF can't be fine-tuned, LoRA adapter repo has VRAM issues on T4)
# Our distilled dataset is comprehensive enough to replace SFT+GRPO in one pass
model, tokenizer = FastModel.from_pretrained(
    model_name      = "unsloth/gemma-4-e2b-it",
    max_seq_length  = 2048,
    load_in_4bit    = True,
    full_finetuning = False,
)

# Re-apply PEFT wrapper for continued training
# Uses same r=32 config as your GRPO training
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r            = 32,
    lora_alpha   = 32,
    lora_dropout = 0,
    bias         = "none",
    random_state = 42,
)

model.print_trainable_parameters()
print("✓ Model ready")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

trainable params: 59,719,680 || all params: 5,182,897,696 || trainable%: 1.1522
✓ Model ready


In [5]:

# ============================================================
# CELL 3 — Enhanced System Prompt (Bakes Techniques 1,3,4,5)
# ============================================================
# This prompt IS the distillation target. It embeds:
#   Technique 1: Constrained decoding via strict format rules
#   Technique 3: Static RAG via few-shot exemplars
#   Technique 4: Selective CoT routing via question-type instructions
#   Technique 5: Per-type prompt templates

SYSTEM_PROMPT = """You are PathOS, an expert AI pathologist for H&E stained histopathology analysis.

ABSOLUTE RULES:
1. NEVER hedge. NEVER say "cannot determine", "unable to", "not possible", "uncertain", "not sure", "difficult to". These are BANNED.
2. ALWAYS commit to a specific, definitive answer based on visible features.
3. EVERY response MUST contain exactly one <answer>...</answer> tag.

OUTPUT FORMAT BY QUESTION TYPE:

YES/NO QUESTIONS (starts with is/are/does/do/was/were/has/have/can/could/should):
- First word MUST be "Yes" or "No". No preamble.
- Then: <answer>yes</answer> or <answer>no</answer>
- Then: One sentence of morphological evidence.
- Example: "Yes. <answer>yes</answer> Nuclear pleomorphism with hyperchromatic nuclei visible."

TISSUE IDENTIFICATION (mentions tissue/patch/classify/cell type/stain):
- List 2-3 key morphological features.
- State tissue classification.
- End with <answer>tissue name</answer>
- Example: "Key features: irregular glands, nuclear pleomorphism. <answer>colorectal adenocarcinoma</answer>"

OPEN-ENDED QUESTIONS (all other):
- Observe visible structures briefly.
- State your definitive answer.
- End with <answer>your answer</answer>

REFERENCE EXAMPLES:
Q: "Is nuclear pleomorphism present?"
A: Yes. <answer>yes</answer> Nuclear variation in size and shape with hyperchromatic nuclei.

Q: "Are goblet cells visible?"
A: Yes. <answer>yes</answer> Mucus-secreting columnar cells visible in intestinal mucosa.

Q: "Is carcinoma present?" (normal tissue)
A: No. <answer>no</answer> Regular crypt architecture with intact basement membrane.

Q: "What tissue type is present?" (tumor)
A: Key features: irregular glands, nuclear pleomorphism, hyperchromatic nuclei. <answer>colorectal adenocarcinoma</answer>

Q: "What tissue type is present?" (normal)
A: Key features: regular crypts, columnar epithelium, goblet cells. <answer>normal colon mucosa</answer>

Q: "What is the primary finding?"
A: Dense small round cells with condensed nuclei and scant cytoplasm. <answer>lymphocytic infiltrate</answer>

Q: "What structures are visible?"
A: Univacuolated lipid droplets with peripheral compressed nuclei. <answer>adipose tissue</answer>

Q: "Is necrosis present?"
A: Yes. <answer>yes</answer> Ghost cell outlines and karyolysis indicate tumor necrosis."""

In [6]:

# ============================================================
# CELL 4 — Tissue Knowledge Base + Response Builders
# ============================================================
from PIL import Image
from torch.utils.data import Dataset, ConcatDataset
from collections import defaultdict
import random

random.seed(42)
IMAGE_SIZE  = 448
MAX_SEQ_LEN = 2048

LABEL_MAP = {
    0: "ADI", 1: "BACK", 2: "DEB", 3: "LYM",
    4: "MUC", 5: "MUS",  6: "NORM", 7: "STR", 8: "TUM"
}

TISSUE_KB = {
    "TUM": {
        "name": "colorectal adenocarcinoma",
        "features": "irregular glands, nuclear pleomorphism, hyperchromatic nuclei, mitotic figures",
        "clinical": "malignant; requires oncology staging workup",
        "malignant": True,
        "yn_features": {
            "malignancy": True, "nuclear pleomorphism": True,
            "mitotic figures": True, "necrosis": False,
            "goblet cells": False, "desmoplastic stroma": False,
        },
    },
    "LYM": {
        "name": "lymphocytic infiltrate",
        "features": "dense small round cells, condensed nuclei, scant cytoplasm, perivascular cuffing",
        "clinical": "immune response; exclude lymphoma",
        "malignant": False,
        "yn_features": {
            "malignancy": False, "nuclear pleomorphism": False,
            "mitotic figures": False, "lymphocytic infiltrate": True,
        },
    },
    "STR": {
        "name": "cancer-associated stroma",
        "features": "desmoplastic spindle fibroblasts, dense collagen, myofibroblasts",
        "clinical": "tumor microenvironment; associated with invasion",
        "malignant": False,
        "yn_features": {
            "malignancy": False, "desmoplastic stroma": True,
            "nuclear pleomorphism": False, "mitotic figures": False,
        },
    },
    "ADI": {
        "name": "adipose tissue",
        "features": "univacuolated lipid droplets, peripheral nuclei, thin fibrous septa",
        "clinical": "pericolonic fat; assess for infiltration",
        "malignant": False,
        "yn_features": {
            "malignancy": False, "nuclear pleomorphism": False,
        },
    },
    "MUC": {
        "name": "mucinous component",
        "features": "extracellular mucin pools, floating epithelial clusters, signet ring cells",
        "clinical": "mucinous adenocarcinoma if tumor-associated",
        "malignant": False,
        "yn_features": {
            "malignancy": False, "mucin": True,
        },
    },
    "MUS": {
        "name": "smooth muscle (muscularis)",
        "features": "elongated spindle cells, cigar-shaped nuclei, intersecting fascicles",
        "clinical": "muscularis propria — critical for invasion staging",
        "malignant": False,
        "yn_features": {
            "malignancy": False, "nuclear pleomorphism": False,
        },
    },
    "NORM": {
        "name": "normal colon mucosa",
        "features": "regular crypts, columnar epithelium, goblet cells, intact basement membrane",
        "clinical": "no pathological abnormality",
        "malignant": False,
        "yn_features": {
            "malignancy": False, "goblet cells": True,
            "nuclear pleomorphism": False, "mitotic figures": False,
        },
    },
    "DEB": {
        "name": "necrotic debris",
        "features": "ghost cell outlines, karyolysis, karyorrhexis, acellular eosinophilic material",
        "clinical": "tumor necrosis; common in high-grade tumors",
        "malignant": False,
        "yn_features": {
            "malignancy": False, "necrosis": True,
        },
    },
    "BACK": {
        "name": "background (non-tissue)",
        "features": "glass slide background, no cellular elements",
        "clinical": "non-diagnostic region",
        "malignant": False,
        "yn_features": {
            "malignancy": False,
        },
    },
}

# --- Question pools ---
TISSUE_QUESTIONS = [
    "What tissue type is present in this histopathology patch?",
    "Identify the tissue class in this H&E stained section.",
    "Classify the tissue type and describe its features.",
    "What does this digitized slide patch show?",
    "What are the key histological features visible here?",
]

YN_QUESTIONS = [
    ("Is malignancy present?", "malignancy"),
    ("Is nuclear pleomorphism present?", "nuclear pleomorphism"),
    ("Are mitotic figures visible?", "mitotic figures"),
    ("Is necrosis present?", "necrosis"),
    ("Are goblet cells visible?", "goblet cells"),
    ("Is desmoplastic stroma present?", "desmoplastic stroma"),
    ("Is lymphocytic infiltrate present?", "lymphocytic infiltrate"),
    ("Does this show a benign lesion?", "benign"),
    ("Is mucin present?", "mucin"),
]


def wrap_answer(ans):
    return f"<answer>{ans.strip()}</answer>"


# --- Build perfect responses ---
def make_tissue_response(label_key):
    """Perfect tissue identification response with <answer> tag."""
    t = TISSUE_KB[label_key]
    return (
        f"Key features: {t['features']}. "
        f"Clinical significance: {t['clinical']}. "
        f"{wrap_answer(t['name'])}"
    )


def make_yn_response(label_key, feature_key):
    """Perfect yes/no response with <answer> tag."""
    t = TISSUE_KB[label_key]
    yn_map = t.get("yn_features", {})

    if feature_key == "benign":
        is_yes = not t["malignant"]
    else:
        is_yes = yn_map.get(feature_key, False)

    if is_yes:
        return f"Yes. {wrap_answer('yes')} The histological features of {t['name']} confirm this finding."
    else:
        return f"No. {wrap_answer('no')} The features of {t['name']} do not support this finding."


def make_open_response(label_key):
    """Perfect open-ended response."""
    t = TISSUE_KB[label_key]
    return (
        f"The visible structures show {t['features']}. "
        f"This indicates {t['name']}. "
        f"{wrap_answer(t['name'])}"
    )


def make_pvqa_yn_response(answer):
    """Perfect PathVQA yes/no response."""
    ans = answer.strip().lower()
    if ans == "yes":
        return f"Yes. {wrap_answer('yes')} The histological features in this image confirm this finding."
    else:
        return f"No. {wrap_answer('no')} The histological features in this image do not support this finding."


def make_pvqa_open_response(answer):
    """Perfect PathVQA open-ended response."""
    ans = answer.strip()
    return (
        f"Based on the visible morphological features, the finding is identifiable. "
        f"{wrap_answer(ans)}"
    )


# --- Image + tokenization helpers ---
def resize(img, size=IMAGE_SIZE):
    if img.mode != "RGB":
        img = img.convert("RGB")
    return img.resize((size, size), Image.LANCZOS)


def build_and_tokenize(img, user_text, assistant_text, tok, max_length=MAX_SEQ_LEN):
    """Build chat messages and tokenize with proper label masking."""
    messages_full = [
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": user_text},
        ]},
        {"role": "assistant", "content": [
            {"type": "text", "text": assistant_text},
        ]},
    ]
    messages_prompt = [
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": user_text},
        ]},
    ]

    full_text = tok.apply_chat_template(
        messages_full, tokenize=False, add_generation_prompt=False
    )
    prompt_text = tok.apply_chat_template(
        messages_prompt, tokenize=False, add_generation_prompt=True
    )

    enc_full = tok(
        text=full_text, images=[img],
        truncation=True, max_length=max_length,
        padding=False, return_tensors="pt",
    )
    enc_prompt = tok(
        text=prompt_text, images=[img],
        truncation=True, max_length=max_length,
        padding=False, return_tensors="pt",
    )

    out = {}
    for k, v in enc_full.items():
        if isinstance(v, torch.Tensor):
            out[k] = v.squeeze(0)
        elif isinstance(v, list):
            out[k] = v[0] if (v and isinstance(v[0], list)) else v
        else:
            out[k] = v

    prompt_len = enc_prompt["input_ids"].shape[1]
    input_ids = out["input_ids"]
    labels = input_ids.clone()
    labels[:prompt_len] = -100  # mask prompt, train only on assistant response
    out["labels"] = labels
    return out

In [7]:

# ============================================================
# CELL 5 — Dataset Classes
# ============================================================

class NCTTissueDataset(Dataset):
    """NCT-CRC tissue identification questions."""
    def __init__(self, dataset, pairs, tok):
        self.dataset, self.pairs, self.tok = dataset, pairs, tok
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        data_idx, label = self.pairs[idx]
        img = resize(self.dataset[data_idx]["image"])
        return build_and_tokenize(
            img,
            f"{SYSTEM_PROMPT}\n\nQuestion: {random.choice(TISSUE_QUESTIONS)}",
            make_tissue_response(label),
            self.tok,
        )


class NCTYNDataset(Dataset):
    """NCT-CRC yes/no questions with ground-truth answers."""
    def __init__(self, dataset, pairs, tok):
        self.dataset, self.pairs, self.tok = dataset, pairs, tok
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        data_idx, label = self.pairs[idx]
        img = resize(self.dataset[data_idx]["image"])
        question, feature = random.choice(YN_QUESTIONS)
        return build_and_tokenize(
            img,
            f"{SYSTEM_PROMPT}\n\nQuestion: {question}",
            make_yn_response(label, feature),
            self.tok,
        )


class NCTOpenDataset(Dataset):
    """NCT-CRC open-ended questions."""
    def __init__(self, dataset, pairs, tok):
        self.dataset, self.pairs, self.tok = dataset, pairs, tok
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        data_idx, label = self.pairs[idx]
        img = resize(self.dataset[data_idx]["image"])
        return build_and_tokenize(
            img,
            f"{SYSTEM_PROMPT}\n\nQuestion: What is the primary clinical finding and its significance?",
            make_open_response(label),
            self.tok,
        )


class PathVQADistillDataset(Dataset):
    """PathVQA with perfectly formatted responses."""
    def __init__(self, dataset, indices, tok):
        self.dataset, self.indices, self.tok = dataset, indices, tok
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        sample = self.dataset[self.indices[idx]]
        answer = sample["answer"].strip()
        is_yn = answer.lower() in ["yes", "no"]
        img = resize(sample["image"])
        ans_text = make_pvqa_yn_response(answer) if is_yn else make_pvqa_open_response(answer)
        return build_and_tokenize(
            img,
            f"{SYSTEM_PROMPT}\n\nQuestion: {sample['question']}",
            ans_text,
            self.tok,
        )


class QuiltDistillDataset(Dataset):
    """QUILT-LLaVA with <answer> tag enforcement."""
    def __init__(self, samples, tok, placeholder_img):
        self.samples, self.tok, self.placeholder = samples, tok, placeholder_img
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        convs = s.get("conversations", [])
        if len(convs) < 2:
            convs = [
                {"from": "human", "value": "Describe this histopathology image."},
                {"from": "gpt", "value": "This image shows histopathological tissue."},
            ]
        user_text = convs[0]["value"].replace("<image>", "").strip()
        model_text = convs[1]["value"].strip()
        # Ensure <answer> tag
        if "<answer>" not in model_text:
            short = model_text.split(".")[0].strip() if "." in model_text else model_text[:50]
            model_text = f"{model_text}\n{wrap_answer(short)}"
        return build_and_tokenize(
            self.placeholder,
            f"{SYSTEM_PROMPT}\n\n{user_text}",
            model_text,
            self.tok,
        )


print("✓ Dataset classes defined")

✓ Dataset classes defined


In [8]:

# ============================================================
# CELL 6 — Load All Datasets + Build Distillation Set
# ============================================================
from datasets import load_dataset

# --- NCT-CRC ---
print("Loading NCT-CRC-HE-100K...")
nct = load_dataset("1aurent/NCT-CRC-HE", split="NCT_CRC_HE_100K",
                    cache_dir="/tmp/hf_datasets_cache")
print(f"  Loaded {len(nct)} samples")

class_buckets = defaultdict(list)
for i, sample in enumerate(nct):
    class_buckets[LABEL_MAP[sample["label"]]].append(i)

# 200 per class for tissue + 200 for YN + 100 for open = 500/class × 9 = 4500
tissue_pairs, yn_pairs, open_pairs = [], [], []
for label, indices in class_buckets.items():
    random.shuffle(indices)
    tissue_pairs.extend([(i, label) for i in indices[:200]])
    yn_pairs.extend([(i, label) for i in indices[200:400]])
    open_pairs.extend([(i, label) for i in indices[400:500]])

random.shuffle(tissue_pairs)
random.shuffle(yn_pairs)
random.shuffle(open_pairs)
print(f"  NCT tissue: {len(tissue_pairs)}, YN: {len(yn_pairs)}, open: {len(open_pairs)}")

# --- PathVQA ---
print("\nLoading PathVQA...")
pvqa = load_dataset("flaviagiammarino/path-vqa", cache_dir="/tmp/hf_datasets_cache")
train_pvqa = pvqa["train"]

def is_quality(s):
    a = s["answer"].strip().lower()
    return len(a) >= 2 and a not in ["n/a", "none", "unknown", "na"]

all_idx = [i for i in range(len(train_pvqa)) if is_quality(train_pvqa[i])]
yn_idx = [i for i in all_idx if train_pvqa[i]["answer"].lower() in ["yes", "no"]]
open_idx = [i for i in all_idx if train_pvqa[i]["answer"].lower() not in ["yes", "no"]]

random.shuffle(yn_idx)
random.shuffle(open_idx)

# Heavy YN emphasis — this is where accuracy matters most
pvqa_selected = yn_idx[:3500] + open_idx[:2000]
random.shuffle(pvqa_selected)
print(f"  PathVQA: {len(pvqa_selected)} (3500 yn + 2000 open)")

# --- QUILT-LLaVA ---
print("\nLoading QUILT-LLaVA-Instruct-107K...")
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN_read")
login(token=hf_token)
quilt_ds = load_dataset(
    "wisdomik/QUILT-LLaVA-Instruct-107K",
    data_files={"train": "quilt_instruct_107k.json"},
    split="train",
    cache_dir="/tmp/hf_datasets_cache",
)

def is_valid_quilt(sample):
    convs = sample.get("conversations", [])
    if len(convs) < 2:
        return False
    return (convs[0].get("from") == "human" and
            convs[1].get("from") == "gpt" and
            len(convs[1].get("value", "").strip()) > 5)

quilt_ds = quilt_ds.filter(is_valid_quilt)
quilt_all = list(quilt_ds)
random.shuffle(quilt_all)
quilt_selected = quilt_all[:4000]
print(f"  QUILT: {len(quilt_selected)}")

# --- Build datasets ---
placeholder_img = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), color=(128, 128, 128))

nct_tissue_ds = NCTTissueDataset(nct, tissue_pairs, tokenizer)
nct_yn_ds     = NCTYNDataset(nct, yn_pairs, tokenizer)
nct_open_ds   = NCTOpenDataset(nct, open_pairs, tokenizer)
pvqa_ds       = PathVQADistillDataset(train_pvqa, pvqa_selected, tokenizer)
quilt_ds_obj  = QuiltDistillDataset(quilt_selected, tokenizer, placeholder_img)

# PathVQA appears twice for emphasis (it's the evaluation benchmark)
combined_ds = ConcatDataset([
    nct_tissue_ds,    # 1800 tissue ID
    nct_yn_ds,        # 1800 yes/no
    nct_open_ds,      # 900 open
    pvqa_ds,          # 5500 PathVQA
    pvqa_ds,          # 5500 PathVQA (×2 emphasis)
    quilt_ds_obj,     # 4000 QUILT reasoning
])

print(f"\n{'='*55}")
print(f"  DISTILLATION DATASET")
print(f"{'='*55}")
print(f"  NCT tissue ID:    {len(nct_tissue_ds)}")
print(f"  NCT yes/no:       {len(nct_yn_ds)}")
print(f"  NCT open:         {len(nct_open_ds)}")
print(f"  PathVQA (×2):     {len(pvqa_ds) * 2}")
print(f"  QUILT-Instruct:   {len(quilt_ds_obj)}")
print(f"  TOTAL:            {len(combined_ds)}")
print(f"{'='*55}")

# Verify
for name, ds in [("NCT tissue", nct_tissue_ds), ("NCT YN", nct_yn_ds), ("PathVQA", pvqa_ds)]:
    s = ds[0]
    shapes = {k: tuple(v.shape) for k, v in s.items() if hasattr(v, "shape")}
    print(f"  {name}: {shapes}")

print("✓ Distillation dataset ready")


Loading NCT-CRC-HE-100K...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

...

README.md: 0.00B [00:00, ?B/s]

quilt_instruct_107k.json:   0%|          | 0.00/189M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/107131 [00:00<?, ? examples/s]

  QUILT: 4000

  DISTILLATION DATASET
  NCT tissue ID:    1800
  NCT yes/no:       1800
  NCT open:         900
  PathVQA (×2):     11000
  QUILT-Instruct:   4000
  TOTAL:            19500
  NCT tissue: {'input_ids': (902,), 'attention_mask': (902,), 'mm_token_type_ids': (902,), 'pixel_values': (2520, 768), 'image_position_ids': (2520, 2), 'labels': (902,)}
  NCT YN: {'input_ids': (893,), 'attention_mask': (893,), 'mm_token_type_ids': (893,), 'pixel_values': (2520, 768), 'image_position_ids': (2520, 2), 'labels': (893,)}
  PathVQA: {'input_ids': (883,), 'attention_mask': (883,), 'mm_token_type_ids': (883,), 'pixel_values': (2520, 768), 'image_position_ids': (2520, 2), 'labels': (883,)}
✓ Distillation dataset ready


In [9]:

# ============================================================
# CELL 7 — Collate Function
# ============================================================
import torch
from torch.nn.utils.rnn import pad_sequence

PAD_ID = tokenizer.pad_token_id or 0

SEQ_PAD = {
    "input_ids": PAD_ID, "attention_mask": 0,
    "labels": -100, "token_type_ids": 0, "mm_token_type_ids": 0,
}

def collate_fn(batch):
    out = {}
    keys = batch[0].keys()
    for k in keys:
        vals = [b[k] for b in batch]
        if k in SEQ_PAD:
            pad_val = SEQ_PAD[k]
            seqs = []
            for v in vals:
                if isinstance(v, torch.Tensor):
                    seqs.append(v.flatten().tolist())
                elif isinstance(v, list):
                    flat = v[0] if (v and isinstance(v[0], list)) else v
                    seqs.append([int(x) for x in flat])
                else:
                    seqs.append([int(v)])
            max_len = max(len(s) for s in seqs)
            padded = [s + [pad_val] * (max_len - len(s)) for s in seqs]
            out[k] = torch.tensor(padded, dtype=torch.long)
            continue
        if k in ("pixel_values", "image_position_ids"):
            try:
                tensors = []
                for v in vals:
                    t = v.squeeze(0) if isinstance(v, torch.Tensor) and v.dim() > 2 else (v if isinstance(v, torch.Tensor) else torch.tensor(v))
                    tensors.append(t)
                out[k] = torch.stack(tensors)
            except Exception:
                out[k] = vals
            continue
        try:
            if isinstance(vals[0], torch.Tensor):
                out[k] = torch.stack([v.squeeze(0) if v.dim() > 0 and v.shape[0] == 1 else v for v in vals])
            elif isinstance(vals[0], (int, float)):
                out[k] = torch.tensor(vals)
            else:
                out[k] = vals
        except Exception:
            out[k] = vals
    return out

print("✓ Collate function defined")

✓ Collate function defined


In [10]:

# ============================================================
# CELL 8 — Phase 1: SFT on Distilled Data
# ============================================================
from trl import SFTTrainer, SFTConfig
from torch.utils.data import DataLoader, RandomSampler
from transformers.trainer_utils import seed_worker

class PathOSSFTTrainer(SFTTrainer):
    def get_train_dataloader(self) -> DataLoader:
        return DataLoader(
            self.train_dataset,
            batch_size=2, sampler=RandomSampler(self.train_dataset),
            collate_fn=collate_fn, num_workers=0,
            pin_memory=False, drop_last=False,
            worker_init_fn=seed_worker,
        )

FastModel.for_training(model)

sft_config = SFTConfig(
    output_dir="/tmp/pathos_distill_sft",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,  # effective batch = 32
    num_train_epochs=1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=25,
    save_strategy="epoch",
    save_total_limit=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,
    dataloader_num_workers=0,
    report_to="none",
    dataset_kwargs={"skip_prepare_dataset": True},
    max_seq_length=MAX_SEQ_LEN,
    remove_unused_columns=False,
)

sft_trainer = PathOSSFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=combined_ds,
    data_collator=collate_fn,
    args=sft_config,
)

print(f"Starting Phase 1: SFT on {len(combined_ds)} distilled samples...")
sft_trainer.train()
print("✓ Phase 1 SFT complete")

model.save_pretrained("/tmp/pathos_distill_sft_ckpt")
tokenizer.save_pretrained("/tmp/pathos_distill_sft_ckpt")
print("✓ SFT checkpoint saved")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


Starting Phase 1: SFT on 19500 distilled samples...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 19,500 | Num Epochs = 1 | Total steps = 610
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 16 x 1) = 32
 "-____-"     Trainable parameters = 59,719,680 of 5,182,897,696 (1.15% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


...

Step,Training Loss
25,0.655020
50,0.260171
75,0.169602
100,0.124900
125,0.106389
150,0.089159
175,0.079600
200,0.079317
225,0.073763
250,0.071100


/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
Unsloth: Restored added_tokens_decoder metadata in /tmp/pathos_distill_sft/checkpoint-610/tokenizer_config.json.


✓ Phase 1 SFT complete


Unsloth: Restored added_tokens_decoder metadata in /tmp/pathos_distill_sft_ckpt/tokenizer_config.json.


✓ SFT checkpoint saved


In [11]:
# ============================================================
# CELL 9 — Phase 2: GRPO (Anti-Hedge + Format Enforcement)
# ============================================================
import re
from trl import GRPOTrainer, GRPOConfig

# --- Reward functions ---
HEDGE_PATTERNS = re.compile(
    r"\b(cannot|can't|unable|not possible|not sure|difficult to|"
    r"hard to|uncertain|cannot determine|cannot confirm|"
    r"i'm not able|i am not able)\b",
    re.IGNORECASE,
)

def extract_answer_tag(text):
    m = re.search(r"<answer>(.*?)</answer>", text, re.IGNORECASE | re.DOTALL)
    return m.group(1).strip().lower() if m else None

def reward_format(completions, **kwargs):
    """+ 0.3 if <answer> tag present, -0.5 if missing."""
    rewards = []
    for c in completions:
        text = c[0]["content"] if isinstance(c, list) else c
        extracted = extract_answer_tag(text)
        rewards.append(0.3 if extracted and len(extracted) > 0 else -0.5)
    return rewards

def reward_no_hedge(completions, **kwargs):
    """-1.0 if hedging language detected."""
    rewards = []
    for c in completions:
        text = c[0]["content"] if isinstance(c, list) else c
        rewards.append(-1.0 if HEDGE_PATTERNS.search(text) else 0.0)
    return rewards

def reward_yn_accuracy(completions, ground_truth=None, question_type=None, **kwargs):
    """+1.0 for correct YN answer, -0.5 for wrong."""
    rewards = []
    for i, c in enumerate(completions):
        if question_type is not None and question_type[i] != "yn":
            rewards.append(0.0); continue
        text = c[0]["content"] if isinstance(c, list) else c
        extracted = extract_answer_tag(text)
        gt = ground_truth[i].lower() if ground_truth else None
        if extracted is None or gt is None:
            rewards.append(0.0)
        elif extracted == gt:
            rewards.append(1.0)
        elif extracted in ["yes", "no"] and extracted != gt:
            rewards.append(-0.5)
        else:
            rewards.append(0.0)
    return rewards

def reward_open_accuracy(completions, ground_truth=None, question_type=None, **kwargs):
    """+0.5 for semantic match on open-ended."""
    rewards = []
    for i, c in enumerate(completions):
        if question_type is not None and question_type[i] != "open":
            rewards.append(0.0); continue
        text = c[0]["content"] if isinstance(c, list) else c
        extracted = extract_answer_tag(text) or text.lower()
        gt = ground_truth[i].lower() if ground_truth else ""
        gt_words = [w for w in gt.split() if len(w) > 3]
        if gt_words and any(w in extracted for w in gt_words):
            rewards.append(0.5)
        elif gt in extracted:
            rewards.append(0.5)
        else:
            rewards.append(0.0)
    return rewards

# --- GRPO dataset from PathVQA validation ---
pvqa_val = pvqa["validation"]

def is_quality_val(s):
    a = s["answer"].strip().lower()
    return len(a) >= 2 and a not in ["n/a", "none", "unknown", "na"]

val_all_idx = [i for i in range(len(pvqa_val)) if is_quality_val(pvqa_val[i])]
val_yn_idx = [i for i in val_all_idx if pvqa_val[i]["answer"].lower() in ["yes", "no"]]
val_open_idx = [i for i in val_all_idx if pvqa_val[i]["answer"].lower() not in ["yes", "no"]]

random.shuffle(val_yn_idx)
random.shuffle(val_open_idx)

grpo_indices = val_yn_idx[:140] + val_open_idx[:160]
random.shuffle(grpo_indices)

def build_grpo_sample(idx):
    sample = pvqa_val[idx]
    img = resize(sample["image"])
    is_yn = sample["answer"].lower() in ["yes", "no"]
    prompt = [{
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": f"{SYSTEM_PROMPT}\n\nQuestion: {sample['question']}"},
        ],
    }]
    return {
        "prompt": prompt,
        "ground_truth": sample["answer"].strip().lower(),
        "question_type": "yn" if is_yn else "open",
    }

grpo_dataset = [build_grpo_sample(i) for i in grpo_indices]
from datasets import Dataset as HFDataset
grpo_hf = HFDataset.from_list(grpo_dataset)

grpo_config = GRPOConfig(
    output_dir="/tmp/pathos_distill_grpo",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    optim="adamw_8bit",
    report_to="none",
    dataloader_num_workers=0,
    remove_unused_columns=False,
    num_generations=4,
    max_completion_length=150,
    temperature=0.7,
    beta=0.04,
    loss_type="grpo",
)

grpo_trainer = GRPOTrainer(
    model=model, tokenizer=tokenizer,
    reward_funcs=[reward_format, reward_no_hedge, reward_yn_accuracy, reward_open_accuracy],
    args=grpo_config,
    train_dataset=grpo_hf,
)

print(f"\nStarting Phase 2: GRPO on {len(grpo_dataset)} samples...")
grpo_trainer.train()
print("✓ Phase 2 GRPO complete")

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))



Starting Phase 2: GRPO on 300 samples...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 300 | Num Epochs = 1 | Total steps = 300
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 59,719,680 of 5,182,897,696 (1.15% trained)
Passing `generation_config` together with generation-related arguments=({'disable_compile', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_format / mean,rewards / reward_format / std,rewards / reward_no_hedge / mean,rewards / reward_no_hedge / std,rewards / reward_yn_accuracy / mean,rewards / reward_yn_accuracy / std,rewards / reward_open_accuracy / mean,rewards / reward_open_accuracy / std
10,929271.500000,1.050000,0.000000,27.175000,25.800000,29.200000,0.000000,27.175000,25.800000,29.200000,23231785.155135,0.300000,0.000000,0.000000,0.000000,0.750000,0.000000,0.000000,0.000000
20,15057.025000,0.625000,0.086603,29.700000,27.100000,33.100000,0.000000,29.700000,27.100000,33.100000,376425.696855,0.300000,0.000000,0.000000,0.000000,0.325000,0.086603,0.000000,0.000000
30,476636.250000,0.375000,0.050000,35.050000,31.500000,39.400000,0.000000,35.050000,31.500000,39.400000,11915906.622974,0.300000,0.000000,-0.075000,0.050000,0.150000,0.000000,0.000000,0.000000
40,211243.675000,0.425000,0.086603,42.000000,34.800000,52.700000,0.000000,42.000000,34.800000,52.700000,5281092.364145,0.300000,0.000000,0.000000,0.000000,0.075000,0.086603,0.050000,0.000000
50,138.502393,0.450000,0.000000,31.875000,28.200000,35.000000,0.000000,31.875000,28.200000,35.000000,3462.559371,0.300000,0.000000,0.000000,0.000000,0.150000,0.000000,0.000000,0.000000
60,2.626841,0.500000,0.000000,32.350000,28.300000,40.800000,0.000000,32.350000,28.300000,40.800000,65.671018,0.300000,0.000000,0.000000,0.000000,0.200000,0.000000,0.000000,0.000000
70,0.608615,0.900000,0.000000,27.900000,24.700000,30.400000,0.000000,27.900000,24.700000,30.400000,15.215372,0.300000,0.000000,0.000000,0.000000,0.600000,0.000000,0.000000,0.000000
80,179.875952,0.250000,0.100000,34.975000,30.300000,40.300000,0.000000,34.975000,30.300000,40.300000,4496.899696,0.300000,0.000000,-0.050000,0.100000,0.000000,0.000000,0.000000,0.000000
90,705.624170,0.375000,0.028868,34.625000,30.000000,38.600000,0.000000,34.625000,30.000000,38.600000,17640.606469,0.300000,0.000000,0.000000,0.000000,0.050000,0.000000,0.025000,0.028868
100,0.828979,0.562500,0.103868,29.450000,26.600000,34.500000,0.000000,29.450000,26.600000,34.500000,20.724484,0.300000,0.000000,-0.025000,0.050000,0.250000,0.000000,0.037500,0.053868


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Unsloth: Restored added_tokens_decoder metadata in /tmp/pathos_distill_grpo/checkpoint-300/tokenizer_config.json.


✓ Phase 2 GRPO complete


In [12]:
# ============================================================
# CELL 10 — Save + Push to HuggingFace
# ============================================================
FINAL_DIR = "/tmp/pathos_distilled_final"
model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f"✓ Final model saved to {FINAL_DIR}")

# Push to HF
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
login(token=hf_token)

HF_REPO = "dhairyapandya/pathos-gemma4-distilled-rl"

print(f"Pushing to {HF_REPO}...")
model.push_to_hub(HF_REPO, token=hf_token)
tokenizer.push_to_hub(HF_REPO, token=hf_token)
print(f"✓ Published → https://huggingface.co/{HF_REPO}")
print(f"\nGGUF conversion: https://huggingface.co/spaces/ggml-org/gguf-my-repo")

Unsloth: Restored added_tokens_decoder metadata in /tmp/pathos_distilled_final/tokenizer_config.json.


✓ Final model saved to /tmp/pathos_distilled_final
Pushing to dhairyapandya/pathos-gemma4-distilled-rl...


README.md:   0%|          | 0.00/575 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/dhairyapandya/pathos-gemma4-distilled-rl


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpi_i8l3a3/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✓ Published → https://huggingface.co/dhairyapandya/pathos-gemma4-distilled-rl

GGUF conversion: https://huggingface.co/spaces/ggml-org/gguf-my-repo
